In [1]:
# Cambia aquí la tabla objetivo ('scada_raw' o 'scada_proc')
tabla_objetivo = "scada_raw"  

# Columnas críticas que nunca se deben borrar
columnas_protegidas = ["plant_code", "Date"]

# --- 3. CONFIGURACIÓN MANUAL FIJA ---
# Escribe aquí directamente las columnas entre comillas. Déjalo como [] si no quieres nada.
columnas_extra_anadir = [] 
columnas_manual_borrar = []

In [2]:
import pandas as pd
import openpyxl

# ==========================================
# --- 1. CONFIGURACIÓN GENERAL ---
# ==========================================
ruta_excel = "/lakehouse/default/Files/Raw PVSyst1/CONFIG/plants_config.xlsx"
nombre_tabla_excel = "T02_sensor_mapping"

# ==========================================
# --- 3. LECTURA Y LÓGICA (No tocar) ---
# ==========================================
if tabla_objetivo == "scada_raw":
    col_excel = "columna_original"
elif tabla_objetivo == "scada_proc":
    col_excel = "columna_destino"
else:
    raise ValueError("La tabla_objetivo debe ser 'scada_raw' o 'scada_proc'")

wb = openpyxl.load_workbook(ruta_excel, data_only=True)
df_excel = None

for ws in wb.worksheets:
    if nombre_tabla_excel in ws.tables:
        rango = ws.tables[nombre_tabla_excel].ref
        datos_brutos = [[celda.value for celda in fila] for fila in ws[rango]]
        df_excel = pd.DataFrame(datos_brutos[1:], columns=datos_brutos[0])
        break

if df_excel is None:
    raise ValueError(f"No se ha encontrado la tabla '{nombre_tabla_excel}' en ninguna hoja del Excel.")

if col_excel not in df_excel.columns:
    raise ValueError(f"La columna '{col_excel}' no existe dentro de la tabla del Excel.")
    
columnas_excel = df_excel[col_excel].dropna().astype(str).str.strip().unique().tolist()

if not spark.catalog.tableExists(tabla_objetivo):
    raise ValueError(f"La tabla '{tabla_objetivo}' no existe en el catálogo.")

columnas_actuales = spark.read.table(tabla_objetivo).columns

# Identificar las que pide el Excel y aún no existen
columnas_faltantes_excel = [c for c in columnas_excel if c not in columnas_actuales]

# Unir lo que falta del Excel con lo que has puesto a mano en la configuración
columnas_a_anadir = list(set(columnas_faltantes_excel + columnas_extra_anadir))

# Filtrar borrados para proteger columnas clave
columnas_a_borrar = [c for c in columnas_manual_borrar if c in columnas_actuales and c not in columnas_protegidas]

print(f"\n=== RESUMEN DE CAMBIOS PARA LA TABLA '{tabla_objetivo}' ===")
print(f" [+] Columnas a AÑADIR (Excel + Manual) ({len(columnas_a_anadir)}): {columnas_a_anadir}")
print(f" [-] Columnas a BORRAR (Solo manual) ({len(columnas_a_borrar)}): {columnas_a_borrar}")
print("========================================================\n")
print("Revisa el resumen y ejecuta la siguiente celda para aplicar los cambios.")

In [3]:
# --- 4. APLICACIÓN DE CAMBIOS ---
if not columnas_a_anadir and not columnas_a_borrar:
    print("No hay cambios que aplicar en la estructura de la tabla.")
else:
    print("\n[PROCESANDO] Aplicando cambios estructurales...")
    
    # Activar el mapeo de columnas para permitir DROP COLUMN en Fabric
    spark.sql(f"ALTER TABLE {tabla_objetivo} SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')")
    
    # Añadir columnas (Excel + Manuales)
    for col in columnas_a_anadir:
        spark.sql(f"ALTER TABLE {tabla_objetivo} ADD COLUMNS (`{col}` DOUBLE)")
        print(f"  [+] Columna '{col}' añadida (rellena de NULLs).")
        
    # Eliminar columnas (Solo las que has pedido a mano)
    for col in columnas_a_borrar:
        spark.sql(f"ALTER TABLE {tabla_objetivo} DROP COLUMN `{col}`")
        print(f"  [-] Columna '{col}' eliminada.")
        
    print(f"\n¡Evolución de esquema completada con éxito para '{tabla_objetivo}'!")
    
    # Limpiar las variables por seguridad
    columnas_a_anadir = []
    columnas_a_borrar = []